# Divergence investigation

Compare pairwise **divergence** across GBOV + GoldenSites (2023 / 2024):

| column | meaning |
|--------|---------|
| `gvf_vs_gcc_div` | satellite GVF vs PhenoCam GCC |
| `gvf_vs_ndvi_div` | satellite GVF vs PhenoCam NDVI |
| `gcc_vs_ndvi_div` | PhenoCam GCC vs PhenoCam NDVI (ground cross-check) |

Divergence is the combined gap+DTW score used elsewhere (lower = better agreement).

Reads existing `anomaly_pipeline/output/metadata/*_scores.csv` (run
`anomaly_check.ipynb` score cells first if those are missing).

**Spin-up** (`gvf_sos == 1`) is dropped by default before plots/tables.

Main figure: grouped bar chart of mean divergence by veg.
Artifacts: `anomaly_pipeline/output/divergence/`.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path.cwd().resolve()
if REPO.name == "anomaly_pipeline":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared.data_collection import load_all_scores, load_table

ANOMALY_DIR = REPO / "anomaly_pipeline" / "output"
METADATA_DIR = ANOMALY_DIR / "metadata"
OUT_DIR = ANOMALY_DIR / "divergence"
OUT_DIR.mkdir(parents=True, exist_ok=True)
UNIFORMITY_JSON = REPO / "uniformity_pipeline" / "output" / "Full" / "step1_uniformity.json"

DIV_COLS = ["gvf_vs_gcc_div", "gvf_vs_ndvi_div", "gcc_vs_ndvi_div"]
DIV_LABELS = {
    "gvf_vs_gcc_div": "GVF vs GCC",
    "gvf_vs_ndvi_div": "GVF vs NDVI",
    "gcc_vs_ndvi_div": "GCC vs NDVI",
}
SOURCES = [
    "GBOV_2023",
    "GBOV_2024",
    "GoldenSites_2023",
    "GoldenSites_2024",
]


## Load scores

Concatenate all metadata score tables. Flag / drop spin-up.


In [ ]:
all_scores = load_all_scores(ANOMALY_DIR)
all_scores = all_scores.loc[all_scores["source"].isin(SOURCES)].copy()
all_scores["spin_up"] = all_scores["gvf_sos"].eq(1.0) if "gvf_sos" in all_scores.columns else False

EXCLUDE_SPINUP = True
pool = all_scores.loc[~all_scores["spin_up"]].copy() if EXCLUDE_SPINUP else all_scores.copy()

print(
    f"rows={len(all_scores)} | spin-up={int(all_scores['spin_up'].sum())} | "
    f"pool={len(pool)} | sources={sorted(pool['source'].unique())}"
)
print("missing divergence counts:")
display(pool[DIV_COLS].isna().sum().to_frame("n_missing"))

show_cols = [
    c for c in [
        "site", "veg", "year", "source",
        *DIV_COLS,
        "gvf_vs_gcc_gap", "gvf_vs_ndvi_gap", "gcc_vs_ndvi_gap",
        "gvf_sos", "spin_up",
    ] if c in pool.columns
]
display(pool[show_cols].head(12))


## Summary tables

Mean / median divergence by **source** and by **veg** (within the clean pool).


In [ ]:
def summarize(df: pd.DataFrame, by: str) -> pd.DataFrame:
    cols = [c for c in DIV_COLS if c in df.columns]
    g = df.groupby(by, dropna=False)[cols]
    out = g.agg(["count", "mean", "median"]).round(3)
    # flatten MultiIndex columns
    out.columns = [f"{a}_{b}" for a, b in out.columns]
    return out.reset_index()

by_source = summarize(pool, "source")
by_veg = summarize(pool, "veg")
by_source_veg = (
    pool.groupby(["source", "veg"], dropna=False)[DIV_COLS]
    .mean()
    .round(3)
    .reset_index()
)

print("By source:")
display(by_source)
print("By veg:")
display(by_veg)
print("By source × veg (mean):")
display(by_source_veg)

by_source.to_csv(OUT_DIR / "divergence_by_source.csv", index=False)
by_veg.to_csv(OUT_DIR / "divergence_by_veg.csv", index=False)
by_source_veg.to_csv(OUT_DIR / "divergence_by_source_veg.csv", index=False)
print(f"Wrote summary CSVs under {OUT_DIR}")


## Grouped bars: mean divergence by veg

For each veg type, three bars = GVF–GCC / GVF–NDVI / GCC–NDVI means.


In [ ]:
def plot_div_bars_by_veg(df: pd.DataFrame, out_png: Path) -> Path:
    vegs = sorted(df["veg"].dropna().unique())
    x = np.arange(len(vegs))
    width = 0.25
    colors = ["#4C78A8", "#F58518", "#54A24B"]

    fig, ax = plt.subplots(figsize=(max(8, 1.2 * len(vegs) + 3), 4.8))
    for i, (col, color) in enumerate(zip(DIV_COLS, colors)):
        means = [
            df.loc[df["veg"].eq(v), col].mean(skipna=True)
            for v in vegs
        ]
        ax.bar(x + (i - 1) * width, means, width, label=DIV_LABELS[col], color=color)

    counts = [int(df["veg"].eq(v).sum()) for v in vegs]
    ax.set_xticks(x)
    ax.set_xticklabels([f"{v}\n(n={n})" for v, n in zip(vegs, counts)])
    ax.set_ylabel("mean divergence")
    ax.set_xlabel("veg")
    ax.set_title("Mean pairwise divergence by veg (all sources)")
    ax.legend(frameon=True)
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Wrote {out_png}")
    return out_png

plot_div_bars_by_veg(pool, OUT_DIR / "divergence_bars_by_veg.png")


## Per-site divergence (AG, DB, EN, GR)

Horizontal grouped bars: each site-year shows all three divergence scores.
GBOV sites are labeled with a `GBOV_` prefix; year is appended to the name.


In [ ]:
VEG_KEEP = ["AG", "DB", "EN", "GR"]
DIV_COLORS = ["#4C78A8", "#F58518", "#54A24B"]


def plot_div_by_site(df: pd.DataFrame, out_png: Path) -> Path:
    """One panel per veg; each site-year has three divergence bars."""
    plot_df = df.loc[df["veg"].isin(VEG_KEEP)].copy()
    plot_df["year_i"] = pd.to_numeric(plot_df["year"], errors="coerce").astype("Int64")
    plot_df["label"] = plot_df["site"].astype(str)
    gbov = plot_df["source"].astype(str).str.startswith("GBOV_")
    plot_df.loc[gbov, "label"] = "GBOV_" + plot_df.loc[gbov, "label"]
    plot_df["label"] = plot_df.apply(
        lambda r: f"{r['label']} {int(r['year_i'])}" if pd.notna(r["year_i"]) else r["label"],
        axis=1,
    )

    vegs = [v for v in VEG_KEEP if v in set(plot_df["veg"])]
    n_veg = max(len(vegs), 1)
    max_n = max((plot_df["veg"].eq(v).sum() for v in vegs), default=4)

    fig, axes = plt.subplots(
        1,
        n_veg,
        figsize=(4.2 * n_veg, max(5.0, 0.38 * max_n + 1.8)),
        sharex=False,
        squeeze=False,
    )

    bar_h = 0.22
    offsets = np.linspace(-(len(DIV_COLS) - 1) / 2, (len(DIV_COLS) - 1) / 2, len(DIV_COLS)) * bar_h

    for ax, veg in zip(axes[0], vegs):
        sub = (
            plot_df.loc[plot_df["veg"].eq(veg)]
            .sort_values(["gvf_vs_ndvi_div", "label"], ascending=[True, True])
            .reset_index(drop=True)
        )
        y = np.arange(len(sub))
        for i, (col, color, dy) in enumerate(zip(DIV_COLS, DIV_COLORS, offsets)):
            vals = sub[col].to_numpy(dtype=float)
            ax.barh(
                y + dy,
                vals,
                height=bar_h * 0.95,
                color=color,
                label=DIV_LABELS[col] if ax is axes[0][0] else None,
                alpha=0.9,
            )

        ax.set_yticks(y)
        ax.set_yticklabels(sub["label"], fontsize=7)
        ax.invert_yaxis()
        ax.set_xlabel("divergence")
        ax.set_title(f"{veg} (n={len(sub)})")
        ax.grid(True, axis="x", alpha=0.3)

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=c, label=DIV_LABELS[col])
        for col, c in zip(DIV_COLS, DIV_COLORS)
    ]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=True, bbox_to_anchor=(0.5, -0.02))
    fig.suptitle("Per-site pairwise divergence (AG / DB / EN / GR)", y=1.01)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Wrote {out_png}")
    return out_png


plot_div_by_site(pool, OUT_DIR / "divergence_by_site_AG_DB_EN_GR.png")


## Site table: divergence + location + surface fractions

One row per scored site-year. Lat/lon and water/urban come from
`uniformity_pipeline/output/Full/step1_uniformity.json` (joined on ROI name + year).

`water%` / `urban%` are shown as **percentages** (same units as the
July 9 WorldCover Progress Report sheet), not 0–1 fractions.


In [ ]:
import json

# join scores → lat/lon/water/urban from step-1 uniformity
uni_sites = json.loads(UNIFORMITY_JSON.read_text())["sites"]
uni = pd.DataFrame(uni_sites)[
    ["name", "year", "lat", "lon", "water_pct", "urban_pct"]
].copy()
uni["year"] = pd.to_numeric(uni["year"], errors="coerce").astype("Int64")

# lat/lon are stable per ROI — fill year gaps from any other year of the same name
latlon = (
    uni.dropna(subset=["lat", "lon"])
    .sort_values("year")
    .drop_duplicates("name", keep="first")[["name", "lat", "lon"]]
    .rename(columns={"lat": "lat_fb", "lon": "lon_fb"})
)

site_table = pool.merge(
    uni,
    left_on=["roi", "year"],
    right_on=["name", "year"],
    how="left",
)
site_table = site_table.merge(latlon, left_on="roi", right_on="name", how="left", suffixes=("", "_dup"))
site_table["lat"] = site_table["lat"].fillna(site_table["lat_fb"])
site_table["lon"] = site_table["lon"].fillna(site_table["lon_fb"])

# WorldCover water/urban in step1 are fractions (0–1); the July 9 sheet reports %
site_table = pd.DataFrame({
    "site_name": site_table["site"],
    "roi": site_table["roi"],
    "veg": site_table["veg"],
    "source": site_table["source"],
    "lat": site_table["lat"],
    "lon": site_table["lon"],
    "year": site_table["year"],
    "gvf_vs_ndvi": site_table["gvf_vs_ndvi_div"],
    "ndvi_vs_gcc": site_table["gcc_vs_ndvi_div"],
    "gcc_vs_gvf": site_table["gvf_vs_gcc_div"],
    "water%": pd.to_numeric(site_table["water_pct"], errors="coerce") * 100.0,
    "urban%": pd.to_numeric(site_table["urban_pct"], errors="coerce") * 100.0,
})

# round for display (pct to 1 decimal like the Progress Report sheet).
# Prefer Python round / format over pandas.round: pandas uses banker's rounding
# (0.05% -> 0.0) while the sheet shows 0.1.
def _round_pct_1dp(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    return x.map(lambda v: float(f"{v:.1f}") if pd.notna(v) else np.nan)

for col in ["lat", "lon", "gvf_vs_ndvi", "ndvi_vs_gcc", "gcc_vs_gvf"]:
    site_table[col] = pd.to_numeric(site_table[col], errors="coerce").round(4)
for col in ["water%", "urban%"]:
    site_table[col] = _round_pct_1dp(site_table[col])

site_table = site_table.sort_values(["veg", "site_name", "year"]).reset_index(drop=True)

out_csv = OUT_DIR / "site_divergence_latlon_fractions.csv"
site_table.to_csv(out_csv, index=False)
print(
    f"rows={len(site_table)} | with lat/lon={site_table['lat'].notna().sum()} | "
    f"with water={site_table['water%'].notna().sum()} | wrote {out_csv}"
)
display(site_table[
    [
        "site_name", "lat", "lon", "year",
        "gvf_vs_ndvi", "ndvi_vs_gcc", "gcc_vs_gvf",
        "water%", "urban%",
    ]
])


## Notes

- Lower divergence = better agreement.
- `gcc_vs_ndvi_div` is the PhenoCam-only baseline; large values mean GCC and NDVI
  already disagree on the ground, so GVF mismatches are harder to interpret.
- Toggle `EXCLUDE_SPINUP` in the load cell if you want spin-up rows included.
